# Unified AI Chatbot — Internship Evaluation

This notebook produces **reproducible evidence** that Tasks 1–6 run end-to-end.
- Uses local fallbacks by default (Gemini disabled)
- Generates sample outputs and simple metrics/plots

In [ ]:
import os
import sys
import platform

# Make Task 2 reproducible/offline: force-disable Gemini (local fallbacks are used).
os.environ.setdefault('DISABLE_GEMINI', '1')

print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())

def _ver(mod_name):
    try:
        m = __import__(mod_name)
        return getattr(m, '__version__', 'unknown')
    except Exception as e:
        return f'not importable ({e})'

for m in ['torch', 'transformers', 'sentence_transformers', 'faiss', 'spacy']:
    print(f'{m}:', _ver(m))

In [ ]:
from chatbot_main import UnifiedChatbot

bot = UnifiedChatbot(lazy_init=True)
print('Bot ready')

## Task 1 — Dynamic Knowledge Base (Retrieval Demo)

In [ ]:
import pandas as pd

queries = [
    'What is Python used for?',
    'Explain vector databases in simple terms.',
    'What is machine learning?',
    'What is data science?',
    'Explain embeddings.'
]
rows = []
for q in queries:
    r = bot.chat(q, task=1)
    rows.append({'query': q, 'response_preview': str(r.get('text',''))[:160].replace('
',' ')})

df = pd.DataFrame(rows)
df

## Task 2 — Multi-Modal (Offline Fallback Demo)

In [ ]:
import tempfile
from PIL import Image

with tempfile.TemporaryDirectory() as td:
    img_path = os.path.join(td, 'solid.png')
    Image.new('RGB', (256, 256), (10, 120, 240)).save(img_path)
    r = bot.chat('What do you see in the image?', task=2, include_images=True, image_paths=[img_path])

r['text']

## Task 3 — Medical Q&A (Sample Outputs)

In [ ]:
for q in ['What are the symptoms of fever?', 'How is hypertension treated?']:
    r = bot.chat(q, task=3)
    print('Q:', q)
    print('A:', str(r.get('text',''))[:500])
    print('-' * 80)

## Task 4 — Domain Expert (Sample Outputs)

In [ ]:
for q in ['Explain transformers in machine learning.', 'Summarize recent trends in graph neural networks.']:
    r = bot.chat(q, task=4)
    print('Q:', q)
    print('A preview:', str(r.get('text',''))[:700])
    print('-' * 80)

## Task 5 — Sentiment (Mini Evaluation + Confusion Matrix)

We evaluate both:
- **VADER label** (derived from compound score)
- **Transformer label** (if available in the environment)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 20-item tiny labeled set (POS/NEG/NEU)
samples = [
    ('I love this so much!', 'POS'),
    ('This is amazing and makes me happy.', 'POS'),
    ('Great job, I am excited.', 'POS'),
    ('I feel calm and relaxed.', 'POS'),
    ('Thank you, that helped a lot.', 'POS'),
    ('I hate this.', 'NEG'),
    ('This is terrible and frustrating.', 'NEG'),
    ('I am stressed and worried.', 'NEG'),
    ('I feel sad and hopeless today.', 'NEG'),
    ('This makes me angry.', 'NEG'),
    ('The sky is blue.', 'NEU'),
    ('I have a meeting at 3pm.', 'NEU'),
    ('The package arrived yesterday.', 'NEU'),
    ('Please update the document.', 'NEU'),
    ('The device is on the table.', 'NEU'),
    ('I am okay.', 'NEU'),
    ('Not bad.', 'NEU'),
    ('I guess it is fine.', 'NEU'),
    ('Could be better.', 'NEU'),
    ('I am not sure.', 'NEU'),
]

labels = ['NEG', 'NEU', 'POS']

def vader_to_label(compound: float) -> str:
    if compound >= 0.05:
        return 'POS'
    if compound <= -0.05:
        return 'NEG'
    return 'NEU'

gold = []
pred_vader = []
pred_trans = []
trans_available = False

for text, y in samples:
    gold.append(y)
    r = bot.chat(text, task=5)

    # Re-run the analyzer directly for raw fields (more reliable than parsing text).
    from modules.sentiment_analysis import SentimentAnalyzer
    analyzer = SentimentAnalyzer(bot.config)
    s = analyzer.analyze(text, user_id='eval')

    scores = s.get('scores') or {}
    pred_vader.append(vader_to_label(float(scores.get('compound', 0.0))))

    t_label = s.get('transformer_label')
    if isinstance(t_label, str) and t_label:
        trans_available = True
        tl = t_label.upper()
        if 'POS' in tl:
            pred_trans.append('POS')
        elif 'NEG' in tl:
            pred_trans.append('NEG')
        else:
            pred_trans.append('NEU')
    else:
        pred_trans.append('NEU')

cm_vader = confusion_matrix(gold, pred_vader, labels=labels)

plt.figure(figsize=(5,4))
sns.heatmap(cm_vader, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.title('Task 5 — VADER Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Gold')
plt.show()

print('VADER report:')
print(classification_report(gold, pred_vader, labels=labels, zero_division=0))

if trans_available:
    cm_t = confusion_matrix(gold, pred_trans, labels=labels)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm_t, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Greens')
    plt.title('Task 5 — Transformer Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Gold')
    plt.show()
    print('Transformer report:')
    print(classification_report(gold, pred_trans, labels=labels, zero_division=0))
else:
    print('Transformer sentiment not available; only VADER evaluated.')

## Task 6 — Multi-language (Detection + Translation Samples)

In [ ]:
phrases = [
    'Translate: Hola, ¿cómo estás?',
    'Translate: Bonjour, comment ça va?',
    'Translate: مرحبا كيف حالك',
    'Translate: 你好，你怎么样？',
    'Translate: I am feeling great today.',
]
for p in phrases:
    r = bot.chat(p, task=6)
    print('Input:', p)
    print(str(r.get('text',''))[:400])
    print('-' * 80)

## Summary

In [ ]:
print('Tasks 1–6 executed in this notebook.')
print('Gemini disabled:', os.environ.get('DISABLE_GEMINI'))
print('If the local arXiv FAISS index is missing/incompatible, Task 4 will fall back to keyword search and/or the live arXiv API.')